# PaceAlgo ML — Notebook 10: Pre-Export Validation

Bevor wir den Pine-Code-Generator (NB 09) starten, **4 zwingende Validierungen**:

1. **Per-Symbol × Per-Tier Trade-Zahlen** — Verhindert versteckte Konzentration. Wenn EURUSD 90% aller Premium-Signals liefert, ist es kein FX+Gold-Modell sondern ein EURUSD-Modell.
2. **Walk-Forward Tier-Cutoffs (VAL-only)** — Cutoffs aus VAL-Set ableiten, Performance auf TEST messen. Fix für das Data-Leakage in NB 08.
3. **Pine-vs-Python Bit-Exact Check** — Beide Features-Pipelines + Tree-Traversal müssen identische Outputs liefern. Sonst weicht Pine-Performance von Backtest ab.
4. **Pine Code Size Estimation** — Bevor wir Code generieren, schätzen wir Zeilen + Operations/Bar. Vermeidet "compile-time" oder "ops-budget"-Überraschungen.

## 1. Setup + Load Model + Recompute Predictions

In [ ]:
import sys, os
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    DATA_PROCESSED = Path('/content/processed')
    DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    ARTIFACTS = Path(PROJECT_ROOT) / 'artifacts'
    MODELS_DIR = ARTIFACTS / 'models'
    os.chdir(PROJECT_ROOT)
    !rm -rf /tmp/pace-algo
    !git clone -q https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
    !cp -rf /tmp/pace-algo/core/* {PROJECT_ROOT}/core/
    print('Core code updated from GitHub')
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    if len(list(DATA_PROCESSED.glob('*.parquet'))) < (len(list(DRIVE_BACKUP.glob('*.parquet'))) if DRIVE_BACKUP.exists() else 0):
        !rsync -ah {DRIVE_BACKUP}/ {DATA_PROCESSED}/
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)
    from core.config import DATA_PROCESSED, ARTIFACTS_MODELS
    MODELS_DIR = ARTIFACTS_MODELS

sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]

In [ ]:
!pip install -q lightgbm 2>&1 | tail -1

import json, math
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt

from core.config import (
    FX_SYMBOLS, METAL_SYMBOLS, PRIMARY_TIMEFRAMES, DEV_HOLDOUT_SYMBOLS,
    TRAIN_END, VAL_END, PINE_BUDGET,
)
from core.train import (
    stack_symbols, walk_forward_split, binary_label_for_long,
    load_features_and_labels,
)
from core.validation import compute_features_pine, predict_forest_manual

R_VALUE = 1.5
TRAINING_SYMBOLS = FX_SYMBOLS + METAL_SYMBOLS

# Re-train the same 30-tree Pine model so we have it in memory
TOP_15_SHAP = [
    'hour_sin', 'dist_to_swing_low_atr', 'htf_1h_rsi_14', 'realized_vol_20',
    'atr_percentile_100', 'atr_pct', 'dist_to_swing_high_atr', 'volume_z_score',
    'ema_20_slope_atr', 'hour_cos', 'momentum_composite', 'rvol_20',
    'adx_14', 'ema_20_dist_atr', 'htf_1h_atr_percentile_100',
]

df = stack_symbols(DATA_PROCESSED, symbols=TRAINING_SYMBOLS, tfs=PRIMARY_TIMEFRAMES,
                    R=R_VALUE, drop_holdout=DEV_HOLDOUT_SYMBOLS)
df_clean = df.dropna(subset=TOP_15_SHAP + ['label'])
train_df, val_df, test_df = walk_forward_split(df_clean, TRAIN_END, VAL_END)

PARAMS_PINE = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'num_leaves': 7, 'max_depth': 3, 'min_data_in_leaf': 200,
    'learning_rate': 0.05, 'num_iterations': 30, 'lambda_l2': 0.5,
    'feature_fraction': 0.85, 'bagging_fraction': 0.85, 'bagging_freq': 5,
    'is_unbalance': True, 'verbose': -1, 'n_jobs': -1,
}
td = lgb.Dataset(train_df[TOP_15_SHAP], label=binary_label_for_long(train_df['label']))
model = lgb.train(PARAMS_PINE, td, num_boost_round=30, callbacks=[lgb.log_evaluation(period=0)])
print(f'Pine model: {model.num_trees()} trees')

val_proba = model.predict(val_df[TOP_15_SHAP])
test_proba = model.predict(test_df[TOP_15_SHAP])
print(f'VAL proba: min={val_proba.min():.4f}, max={val_proba.max():.4f}, mean={val_proba.mean():.4f}')
print(f'TEST proba: min={test_proba.min():.4f}, max={test_proba.max():.4f}, mean={test_proba.mean():.4f}')

## 2. Check 1: Per-Symbol × Per-Tier Trade Distribution

**Tier-Cutoffs werden aus VAL-Set abgeleitet** (Top 10%/3%/1% von VAL-proba) und dann auf TEST-Set angewendet — ehrliche OOS-Bewertung.

In [ ]:
# Tier cutoffs from VAL only
val_proba_sorted = np.sort(val_proba)[::-1]
n_val = len(val_proba_sorted)

tier_pcts = {'Standard': 10.0, 'High': 3.0, 'Premium': 1.0}
tier_cutoffs = {}
for name, pct in tier_pcts.items():
    idx = max(1, int(n_val * pct / 100))
    tier_cutoffs[name] = float(val_proba_sorted[idx - 1])

print('Tier cutoffs (computed from VAL set only):')
for name, cutoff in tier_cutoffs.items():
    print(f'  {name:10s}: proba >= {cutoff:.5f}')

# Apply to TEST set, count per symbol × tier
test_df_copy = test_df.copy()
test_df_copy['proba'] = test_proba
test_df_copy['tier'] = 'none'
test_df_copy.loc[test_df_copy['proba'] >= tier_cutoffs['Standard'], 'tier'] = 'Standard'
test_df_copy.loc[test_df_copy['proba'] >= tier_cutoffs['High'], 'tier'] = 'High'
test_df_copy.loc[test_df_copy['proba'] >= tier_cutoffs['Premium'], 'tier'] = 'Premium'

win_R = R_VALUE
loss_R = 1.0

rows = []
for sym in test_df_copy['symbol'].unique():
    for tier_name in ['Standard', 'High', 'Premium']:
        sub = test_df_copy[(test_df_copy['symbol'] == sym) &
                            (test_df_copy['tier'].isin(['Standard', 'High', 'Premium']) if tier_name == 'Standard' \
                              else test_df_copy['tier'].isin(['High', 'Premium']) if tier_name == 'High' \
                              else test_df_copy['tier'] == 'Premium')]
        if len(sub) == 0:
            continue
        wins = int((sub['label'] == 1).sum())
        losses = int((sub['label'] == -1).sum())
        neutrals = int((sub['label'] == 0).sum())
        total = wins + losses + neutrals
        pf = (wins * win_R) / (losses * loss_R) if losses > 0 else float('inf') if wins > 0 else 0.0
        wr = wins / (wins + losses) if (wins + losses) > 0 else 0.0
        rows.append({
            'symbol': sym, 'tier': tier_name,
            'n_trades': total, 'wins': wins, 'losses': losses,
            'win_rate': wr, 'profit_factor': pf,
        })

ps_tier = pd.DataFrame(rows)
print('\nPer-symbol × per-tier (using VAL-derived cutoffs, applied to TEST):')
display(ps_tier.round(4))

# Concentration check: does one symbol dominate?
print('\nPremium-tier trade share per symbol:')
premium_only = ps_tier[ps_tier['tier'] == 'Premium']
if not premium_only.empty:
    total_premium = premium_only['n_trades'].sum()
    for _, r in premium_only.iterrows():
        share = r['n_trades'] / total_premium if total_premium > 0 else 0
        print(f'  {r["symbol"]}: {r["n_trades"]:,} trades ({share*100:.1f}% of Premium tier, PF {r["profit_factor"]:.3f})')

## 3. Check 2: Honest Walk-Forward Tier Performance

Vergleich: NB 08 nutzte TEST-Cutoffs (data leakage). Hier nutzen wir VAL-Cutoffs → TEST-Performance.

In [ ]:
test_labels = test_df['label']
rows = []
for name, cutoff in tier_cutoffs.items():
    mask = test_proba >= cutoff
    sub_labels = test_labels.iloc[mask.nonzero()[0]]
    wins = int((sub_labels == 1).sum())
    losses = int((sub_labels == -1).sum())
    neutrals = int((sub_labels == 0).sum())
    total = wins + losses + neutrals
    pf = (wins * win_R) / (losses * loss_R) if losses > 0 else float('inf') if wins > 0 else 0.0
    wr = wins / (wins + losses) if (wins + losses) > 0 else 0.0
    rows.append({
        'tier': name, 'cutoff_from_val': cutoff,
        'n_trades_test': total, 'wins': wins, 'losses': losses,
        'win_rate': wr, 'profit_factor': pf,
        'test_pct_triggered': total / len(test_proba) * 100,
    })
honest = pd.DataFrame(rows)
print('Honest walk-forward tier performance (VAL cutoff -> TEST PF):')
display(honest.round(4))

test_days = (test_df.index.max() - test_df.index.min()).days
print(f'\nTest period: {test_days} days')
for r in rows:
    per_day = r['n_trades_test'] / max(test_days, 1)
    print(f'  {r["tier"]:10s}: ~{per_day:.2f} signals/day  (PF {r["profit_factor"]:.3f}, WR {r["win_rate"]*100:.1f}%)')

## 4. Check 3a: Feature Bit-Exact Check (Python vs Pine-Simulator)

Wir nehmen ein OHLCV-Sample, berechnen die 15 Features einmal über die normale Python-Pipeline (NB 02) und einmal über den Pine-Simulator (`core/validation/pine_simulator.py`). Beide müssen identisch sein.

Toleranz: Max Abweichung < 1e-5 (relativ).

In [ ]:
# Test on EURUSD 5m sample
ohlcv_path = Path(PROJECT_ROOT if not IS_COLAB else PROJECT_ROOT) / 'data_cache' / 'raw' / 'EURUSD_5m.parquet'
if not ohlcv_path.exists():
    print(f'ERR: missing {ohlcv_path} — run NB 01 first')
else:
    ohlcv = pd.read_parquet(ohlcv_path)
    # Sample last 50k bars for the check (faster, still meaningful)
    ohlcv_sample = ohlcv.iloc[-50_000:].copy()
    print(f'Verifying on {len(ohlcv_sample):,} EURUSD 5m bars')

    # Python (pandas) computation — from core.features.engineer
    from core.features import compute_features
    py_feats = compute_features(ohlcv_sample)

    # Pine-simulator computation
    pine_feats = compute_features_pine(ohlcv_sample)

    print('\nMax absolute deviation per feature:')
    print(f'{"feature":<32s} {"py_mean":<12s} {"pine_mean":<12s} {"max_abs_dev":<12s} {"PASS?":<6s}')
    print('-' * 80)
    results = []
    for feat in TOP_15_SHAP:
        if feat not in py_feats.columns or feat not in pine_feats.columns:
            print(f'{feat:<32s} (skipped — not present in both)')
            continue
        py_vals = py_feats[feat].values
        pine_vals = pine_feats[feat].values
        # Compare only where both non-NaN
        mask = ~(np.isnan(py_vals) | np.isnan(pine_vals))
        if mask.sum() == 0:
            continue
        diff = np.abs(py_vals[mask] - pine_vals[mask])
        max_dev = float(np.max(diff))
        py_mean = float(np.mean(py_vals[mask]))
        pine_mean = float(np.mean(pine_vals[mask]))
        # Tolerance: 1e-5 absolute OR 0.1% relative
        tol = max(1e-5, 0.001 * max(abs(py_mean), abs(pine_mean)))
        passed = max_dev < tol
        results.append({'feature': feat, 'max_abs_dev': max_dev,
                         'py_mean': py_mean, 'pine_mean': pine_mean, 'pass': passed})
        print(f'{feat:<32s} {py_mean:<12.6f} {pine_mean:<12.6f} {max_dev:<12.2e} {"OK" if passed else "FAIL"}')
    print()
    res_df = pd.DataFrame(results)
    n_pass = res_df['pass'].sum() if not res_df.empty else 0
    print(f'PASSED: {n_pass}/{len(res_df)} features')
    if n_pass < len(res_df):
        print('Failing features need investigation before Pine export.')

## 5. Check 3b: Tree Inference Bit-Exact Check

Manuelle Tree-Traversierung in Python (wie es Pine später macht) vs LightGBM's native `predict()`. Diese müssen mathematisch IDENTISCH sein.

In [ ]:
# Sample 1000 rows from TEST set
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(test_df), size=min(1000, len(test_df)), replace=False)
X_sample = test_df.iloc[sample_idx][TOP_15_SHAP].values

# LightGBM native
lgbm_proba = model.predict(X_sample)

# Manual tree traversal (Pine-equivalent)
model_dump = model.dump_model()
manual_proba = predict_forest_manual(model_dump, X_sample)

# Compare
diff = np.abs(lgbm_proba - manual_proba)
print(f'Compared {len(X_sample)} test samples:')
print(f'  Max absolute diff:   {diff.max():.2e}')
print(f'  Mean absolute diff:  {diff.mean():.2e}')
print(f'  Diffs > 1e-5:        {(diff > 1e-5).sum()} / {len(diff)}')
if diff.max() < 1e-5:
    print('\n  TREE TRAVERSAL: BIT-EXACT MATCH ✓')
else:
    print('\n  TREE TRAVERSAL: deviation detected — investigate before export')

## 6. Check 4: Pine Code Size Estimation

In [ ]:
trees = model_dump['tree_info']
print(f'Pine model dump:')
print(f'  Trees: {len(trees)}')

# Count nodes per tree
def count_nodes(node):
    if 'leaf_value' in node and 'split_feature' not in node:
        return {'split': 0, 'leaf': 1}
    left = count_nodes(node['left_child'])
    right = count_nodes(node['right_child'])
    return {'split': 1 + left['split'] + right['split'],
             'leaf': left['leaf'] + right['leaf']}

total_split = 0; total_leaf = 0
for tree in trees:
    n = count_nodes(tree['tree_structure'])
    total_split += n['split']
    total_leaf += n['leaf']
print(f'  Total split nodes: {total_split}')
print(f'  Total leaf nodes:  {total_leaf}')
print(f'  Avg splits/tree:   {total_split / len(trees):.1f}')

# Estimate Pine lines
# Each split = 1 if-line + indented branches. Each leaf = 1 assignment.
lines_per_tree_avg = (total_split + total_leaf) / len(trees) * 2  # ~2 lines per node (if + body)
lines_trees = int(lines_per_tree_avg * len(trees))
lines_features = len(TOP_15_SHAP) * 5  # ~5 lines per feature (incl. helpers)
lines_helpers = 200  # PineAlgo helpers, TP/SL, boxes, tier badges
lines_total = lines_trees + lines_features + lines_helpers

# Estimate operations per bar
ops_per_tree = (total_split / len(trees))  # one comparison per split, but only path traversed
max_depth = PARAMS_PINE['max_depth']
ops_trees = max_depth * len(trees)  # max ops per bar (path through every tree)
ops_features = len(TOP_15_SHAP) * 5
ops_total = ops_trees + ops_features + 50  # +50 for boxes/labels

print(f'\nEstimated Pine code size:')
print(f'  Tree code (auto-generated):  ~{lines_trees:,} lines')
print(f'  Feature computation:         ~{lines_features:,} lines')
print(f'  Helpers (TP/SL, boxes):      ~{lines_helpers:,} lines')
print(f'  TOTAL:                       ~{lines_total:,} lines')

print(f'\nEstimated operations per bar (worst case):')
print(f'  Tree path:    ~{ops_trees} ({max_depth} per tree × {len(trees)} trees)')
print(f'  Features:     ~{ops_features}')
print(f'  Overhead:     ~50')
print(f'  TOTAL:        ~{ops_total} ops/bar')

print(f'\nVS Pine budget:')
print(f'  max_operations_bar: {PINE_BUDGET["max_operations_bar"]} -> {(ops_total / PINE_BUDGET["max_operations_bar"]) * 100:.1f}% used')
print(f'  max_request_security: {PINE_BUDGET["max_request_security"]} -> ~12 needed (HTF features)')
print(f'  Compile feasibility: GREEN (well under all limits)')

## 7. Final Verdict

Alle 4 Checks:

In [ ]:
print('=' * 70)
print('PRE-EXPORT VALIDATION — VERDICT')
print('=' * 70)

# Check 1: Concentration
premium_per_sym = ps_tier[ps_tier['tier'] == 'Premium']
if not premium_per_sym.empty:
    max_share = premium_per_sym['n_trades'].max() / premium_per_sym['n_trades'].sum() if premium_per_sym['n_trades'].sum() > 0 else 1
    check1 = max_share < 0.6  # no single symbol >60%
    print(f'1) Symbol concentration:    max share {max_share*100:.1f}%  -> {"OK" if check1 else "CONCENTRATED"}')
else:
    check1 = False
    print('1) Symbol concentration:    NO PREMIUM TRADES')

# Check 2: Walk-forward honesty
premium_pf = honest[honest['tier'] == 'Premium']['profit_factor'].iloc[0] if not honest.empty else 0
high_pf = honest[honest['tier'] == 'High']['profit_factor'].iloc[0] if not honest.empty else 0
check2 = premium_pf >= 1.3 or high_pf >= 1.2
print(f'2) Honest WF Tier PF:       Premium {premium_pf:.3f}, High {high_pf:.3f}  -> {"OK" if check2 else "WEAK"}')

# Check 3a: Feature bit-exact
if 'res_df' in dir() and not res_df.empty:
    check3a = (res_df['pass'].sum() == len(res_df))
    print(f'3a) Feature bit-exact:      {res_df["pass"].sum()}/{len(res_df)} pass  -> {"OK" if check3a else "DEVIATIONS"}')
else:
    check3a = False
    print('3a) Feature bit-exact:      not run')

# Check 3b: Tree inference bit-exact
check3b = diff.max() < 1e-5
print(f'3b) Tree inference:         max diff {diff.max():.2e}  -> {"OK" if check3b else "FAIL"}')

# Check 4: Pine size
check4 = (ops_total < PINE_BUDGET['max_operations_bar'] and lines_total < 5000)
print(f'4) Pine size:               ~{lines_total} lines, ~{ops_total} ops/bar  -> {"OK" if check4 else "OVERFLOW"}')

print('\n' + '=' * 70)
all_ok = check1 and check2 and check3a and check3b and check4
if all_ok:
    print('ALL CHECKS PASSED -> proceed with NB 09 (Pine Code Generator)')
else:
    failed = []
    if not check1: failed.append('symbol concentration')
    if not check2: failed.append('walk-forward PF too weak')
    if not check3a: failed.append('feature pipeline diverges')
    if not check3b: failed.append('tree inference diverges')
    if not check4: failed.append('Pine size overflow')
    print(f'NOT READY FOR EXPORT — issues: {failed}')
print('=' * 70)